# Sampler Validation: Marginal Checks

For each validation target, we run the Boomerang,resample uniformly in time, 
and overlay sample histograms against the known marginal densities.

In [ ]:
import os
os.chdir('../..')

import numpy as np
import matplotlib.pyplot as plt

from benchmarks_august.targets.validation_sticky import spike_slab_gaussian, spike_slab_linreg, spike_slab_logreg
from benchmarks_august.samplers.factories import build_sampler
from sazz.samplers.boomerang_sampler.utils import resample_pdmp_path, resample_sticky_pdmp_path
from benchmarks_august.samplers.warmstart import warmup_reference

In [ ]:
# ── Shared settings ──────────────────────────────────────────────
N_SKELETON  = 20000
N_RESAMPLE  = 50000
BURNIN_FRAC = 0.1
refresh_rate = 1.0

def run_and_resample(sampler, target, sticky=True, warmup=True, N_SKELETONS=N_SKELETON):
    """Warmup, preprocess, sample, and return time-uniform resamples."""
    if warmup:
        warmup_reference(sampler, n_rounds=3, n_pilot=500,
                         sticky=sticky, target=target)
    else:
        method = target.meta.get('preprocess_method', 'diagonal')
        if method == 'manual':
            sampler.preprocess(method='manual',
                               x_ref=target.x_ref,
                               Sigma_inv=target.Sigma_inv)
        else:
            sampler.preprocess(method='diagonal')
    
    sampler.reset(N=N_SKELETONS)
    sampler.sample_auto(diagnostics=True)
    
    if sticky:
        _, samples = resample_sticky_pdmp_path(sampler, n_samples=N_RESAMPLE,
                                               burnin_frac=BURNIN_FRAC)
    else:
        _, samples = resample_pdmp_path(sampler, n_samples=N_RESAMPLE,
                                        burnin_frac=BURNIN_FRAC)
    return samples


def plot_marginals(target, samples_dict, figname=None, bins=50):
    """Plot marginal histograms against true densities for each coordinate."""
    marginals = target.meta['marginal_grids']
    D = target.D
    n_samplers = len(samples_dict)

    fig, axes = plt.subplots(n_samplers, D, figsize=(3.5 * D, 3 * n_samplers),
                             squeeze=False)

    colors = ['steelblue', 'darkorange', 'seagreen', 'firebrick']

    for row, (label, samples) in enumerate(samples_dict.items()):
        for col in range(D):
            ax = axes[row, col]
            mg = marginals[col]
            grid, pdf = mg['grid'], mg['pdf']

            x = samples[:, col]
            # Clip histogram range to the reference grid so the overlay aligns
            x_lo, x_hi = float(grid[0]), float(grid[-1])
            ax.hist(x, bins=bins, range=(x_lo, x_hi), density=True, alpha=0.5,
                    color=colors[row % len(colors)],
                    label=label if (row == 0 and col == 0) else None)
            ax.plot(grid, pdf, 'k-', lw=1.5,
                    label='True' if (row == 0 and col == 0) else None)
            ax.set_xlim(x_lo, x_hi)

            # Annotate out-of-range sample mass if nontrivial
            frac_out = np.mean((x < x_lo) | (x > x_hi))
            if frac_out > 0.01:
                ax.text(0.02, 0.95, f"{frac_out:.1%} out of range",
                        transform=ax.transAxes, fontsize=7,
                        va='top', color='firebrick')

            if row == 0:
                ax.set_title(mg['label'])
            if col == 0:
                ax.set_ylabel(label)
            if row == n_samplers - 1:
                ax.set_xlabel(mg['label'])

    # One legend for the whole figure
    handles, labels_ = axes[0, 0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels_, loc='upper right',
                   bbox_to_anchor=(0.99, 0.99), fontsize=9)

    fig.suptitle(target.name, fontsize=14, y=1.02)
    fig.tight_layout()
    if figname:
        fig.savefig(figname, dpi=150, bbox_inches='tight')
    plt.show()  
    

## 1. Gaussian sanity check

In [ ]:
diag_gauss = spike_slab_gaussian(D=5)
ar_gauss = spike_slab_gaussian(D=5, cov="ar1")
random_gauss = spike_slab_gaussian(D=5, cov="random")

target_gauss = random_gauss

s_gauss = build_sampler('sticky_boomerang', target_gauss, N=N_SKELETON, refresh_rate=refresh_rate, kappa=target_gauss.meta['kappa'])
samp_gauss = run_and_resample(s_gauss, target_gauss, warmup=False)

s_gauss_pli = build_sampler('sticky_boomerang_pli', target_gauss, N=N_SKELETON, refresh_rate=refresh_rate, kappa=target_gauss.meta['kappa'])
samp_gauss_pli = run_and_resample(s_gauss_pli, target_gauss, warmup=False)

# Check: should be zero bounces
df = s_gauss.diagnostics_df
n_bounces = df[(df['event_type'] == 'bounce') & (df['accepted'] == True)].shape[0]
n_refresh = df[df['event_type'] == 'refresh'].shape[0]
wall = df['wall_seconds'].sum()
grad_evals = s_gauss.gradient_evals
df_pli = s_gauss_pli.diagnostics_df
n_bounces_pli = df_pli[(df_pli['event_type'] == 'bounce') & (df_pli['accepted'] == True)].shape[0]
n_refresh_pli = df_pli[df_pli['event_type'] == 'refresh'].shape[0]
wall_pli = df_pli['wall_seconds'].sum()
grad_evals_pli = s_gauss_pli.gradient_evals

print("--------- Sticky Boomerang ---------")
print(f"Accepted bounces: {n_bounces}  (expect 0)")
print(f"Refreshments:     {n_refresh}  (expect all skeleton points)")
print(f"Walltime:         {wall}")
print(f"Grad evals per skeleton point: {grad_evals / s_gauss.N:.1f}")
print("--------- Sticky Boomerang PLI ---------")
print(f"Accepted bounces: {n_bounces_pli}  (expect 0)")
print(f"Refreshments:     {n_refresh_pli}  (expect all skeleton points)")
print(f"Walltime:         {wall_pli}")
print(f"Grad evals per skeleton point: {grad_evals_pli / s_gauss_pli.N:.1f}")

plot_marginals(target_gauss, {'Boomerang': samp_gauss,
                              'Boomerang PLI': samp_gauss_pli})
               # ,figname='validation_gaussian_refcheck.pdf')

## 2. Gaussian Regression

In [ ]:
target_regression = spike_slab_linreg(n=100, p=8, sparsity="sparse")


In [ ]:
# Boomerang
s_bb = build_sampler('sticky_boomerang', target_regression, N=N_SKELETON, refresh_rate=0.1, kappa=target_regression.meta['kappa'])
samp_bb = run_and_resample(s_bb, target_regression, warmup=False)

In [ ]:
# Boomerang PLI
s_bb_pli = build_sampler('sticky_boomerang_pli', target_regression, N=N_SKELETON, refresh_rate=0.1, kappa=target_regression.meta['kappa'])
samp_bb_pli = run_and_resample(s_bb_pli, target_regression, warmup=False)

In [ ]:
plot_marginals(target_regression, {
    'Boomerang': samp_bb,
    'Boomerang PLI': samp_bb_pli,
})#, figname='validation_beta_binomial.pdf')


## 3. Rosenbrocks banana

In [ ]:
target_logistic_regression = spike_slab_logreg(n=100, p=8, sparsity="sparse")


In [ ]:
# Boomerang
s_banana = build_sampler('sticky_boomerang', target_logistic_regression, N=N_SKELETON, refresh_rate=0.1, kappa=target_logistic_regression.meta['kappa'])
samp_banana = run_and_resample(s_banana, target_logistic_regression, warmup=False, N_SKELETONS=N_SKELETON)

In [ ]:
# Boomerang PLI
s_banana_pli = build_sampler('sticky_boomerang_pli', target_logistic_regression, N=N_SKELETON, refresh_rate=0.1, kappa=target_logistic_regression.meta['kappa'])
samp_banana_pli = run_and_resample(s_banana_pli, target_logistic_regression, warmup=False, N_SKELETONS=N_SKELETON)

In [ ]:
plot_marginals(target_logistic_regression, {
    'Boomerang': samp_banana,
    'Boomerang PLI': samp_banana_pli,
})#, figname='validation_neals_funnel.pdf')